In [1]:
!pip install -q datasets

In [2]:
from datasets import load_dataset

imdb = load_dataset("imdb")

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [3]:
imdb

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [4]:
train_df = imdb['train'].to_pandas()
test_df = imdb['test'].to_pandas()

In [5]:
train_df[:10]

,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0
5,I would put this at the top of my list of film...,0
6,Whoever wrote the screenplay for this movie ob...,0
7,"When I first saw a glimpse of this movie, I qu...",0
8,"Who are these ""They""- the actors? the filmmake...",0
9,This is said to be a personal film for Peter B...,0


In [6]:
from sklearn.model_selection import train_test_split

train_df, valid_df = train_test_split(train_df, test_size= 0.2, stratify= train_df['label'], random_state = 42)

In [7]:
print(f'Train: {train_df[:10]}')
print("="*80)
print(f'Valid: {valid_df[:10]}')

Train:                                                     text  label
20022  I have always been a huge James Bond fanatic! ...      1
4993   I am a Christian and I say this movie had terr...      0
24760  Neatly sandwiched between THE STRANGER, a smal...      1
13775  Years ago I did follow a soap on TV. So I was ...      1
20504  Here's a gritty, get-the-bad guys revenge stor...      1
16170  This is the best picture about baseball since ...      1
12144  I had the greatest enthusiasm going in to the ...      0
20287  this was one of those $.50 cent deals of yore-...      1
17830  This is a very light headed comedy about a won...      1
19810  Expecting to see a "cute little film" from mai...      1
Valid:                                                     text  label
15626  That 70s Show is the best TV show ever, period...      1
18778  This is a must see for independant movie fans,...      1
8927   After watching this, I had lost a little respe...      0
3998   ...through the simi

In [8]:
print("Train:", train_df.shape)
print("Validation:", valid_df.shape)
print("Test:", test_df.shape)

Train: (20000, 2)
Validation: (5000, 2)
Test: (25000, 2)


### Text Preprocessing

In [9]:
import re
import string

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"<.*?>", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    return text

#### Sample Test to check preprocessing is working perfectly

In [10]:
sample = train_df['text'][0]

print("Before:")
print(sample)

print("\nAfter:")
print(preprocess_text(sample))

Before:
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far betw

#### Now apply preprocess_text function to all

In [11]:
train_df['clean_text'] = train_df['text'].apply(preprocess_text)
valid_df['clean_text'] = valid_df['text'].apply(preprocess_text)
test_df['clean_text'] = test_df['text'].apply(preprocess_text)

In [12]:
train_df[["text", "clean_text", "label"]].head()

,text,clean_text,label
20022,I have always been a huge James Bond fanatic! ...,i have always been a huge james bond fanatic i...,1
4993,I am a Christian and I say this movie had terr...,i am a christian and i say this movie had terr...,0
24760,"Neatly sandwiched between THE STRANGER, a smal...",neatly sandwiched between the stranger a small...,1
13775,Years ago I did follow a soap on TV. So I was ...,years ago i did follow a soap on tv so i was c...,1
20504,"Here's a gritty, get-the-bad guys revenge stor...",heres a gritty getthebad guys revenge story st...,1


### TF-IDF Vectorization

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

X_train = tfidf.fit_transform(train_df['clean_text'])
X_valid = tfidf.transform(valid_df['clean_text'])
X_test = tfidf.transform(test_df['clean_text'])

In [14]:
y_train = train_df['label']
y_valid = valid_df['label']
y_test = test_df['label']

In [15]:
print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("X_test:", X_test.shape)

X_train: (20000, 123871)
X_valid: (5000, 123871)
X_test: (25000, 123871)


In [16]:
print(tfidf.get_feature_names_out()[:20])

['00' '000' '0000000000001' '000001' '0001' '00015' '001' '002' '00383042'
 '006' '007' '0079' '0080' '0083' '00s' '01' '010' '01000' '010guinea'
 '010ps']


In [17]:
print(X_train[0])

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 207 stored elements and shape (1, 123871)>
  Coords	Values
  (0, 49877)	0.06560343694528588
  (0, 5692)	0.05422369707255753
  (0, 11586)	0.0195411202498553
  (0, 53363)	0.07345065675437104
  (0, 58121)	0.11183037246250944
  (0, 14350)	0.2871780130670147
  (0, 39044)	0.06035128321792599
  (0, 96386)	0.15022281924476527
  (0, 5428)	0.027270659051145542
  (0, 5117)	0.014001381492386568
  (0, 77266)	0.07932183304293876
  (0, 108967)	0.10123289313224697
  (0, 40794)	0.021753659006249162
  (0, 37494)	0.0354500398351799
  (0, 42361)	0.0449101163430613
  (0, 29855)	0.03951268981001468
  (0, 6701)	0.024767533626277073
  (0, 27456)	0.02945174366667699
  (0, 6216)	0.12139877103201652
  (0, 121783)	0.05453330364214082
  (0, 57160)	0.09280855415181169
  (0, 76144)	0.02558821583159766
  (0, 35831)	0.05339507288307048
  (0, 47244)	0.052855423895166696
  (0, 37202)	0.02986768480887733
  :	:
  (0, 121634)	0.02523187117418614
  (0, 111392)	0.

### Logistic Regression

In [18]:
### logistic regression model train to check performance after TF-IDF to see how the model performs after TF-IDF

from sklearn.linear_model import LogisticRegression

model = LogisticRegression(random_state=42)

model.fit(X_train, y_train)

LogisticRegression(random_state=42)

In [19]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_predict = model.predict(X_valid)



accuracy = accuracy_score(y_valid, y_predict)
precision = precision_score(y_valid, y_predict)
recall = recall_score(y_valid, y_predict)
f1 = f1_score(y_valid, y_predict)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)


Accuracy : 0.8872
Precision: 0.878125
Recall   : 0.8992
F1 Score : 0.8885375494071146


#### TF-IDF + Logistic Regression timing

In [20]:
import time

start_time = time.time()

tfidf = TfidfVectorizer()

X_train = tfidf.fit_transform(train_df["clean_text"])
X_valid = tfidf.transform(valid_df["clean_text"])
X_test = tfidf.transform(test_df["clean_text"])

model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

tfidf_training_time = time.time() - start_time

print(f"TF-IDF + Logistic Regression training time: {tfidf_training_time:.2f} seconds")

TF-IDF + Logistic Regression training time: 10.84 seconds


### Word2Vec

In [21]:
!pip install -q gensim

In [22]:
### Now as for Word2Vec we need token. So, now we are going to split all words from all reviews

from gensim.models import Word2Vec

train_tokens = train_df["clean_text"].str.split().tolist()
val_tokens = valid_df["clean_text"].str.split().tolist()
test_tokens = test_df["clean_text"].str.split().tolist()

In [23]:
## We can check that first review is splited or not >>
# train_tokens[0]

In [24]:
# Word2Vec config

w2v_model = Word2Vec(
    sentences=train_tokens,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    seed=42
)

In [25]:
## Check the movie word is in 100-dimention or not
w2v_model.wv["movie"]

array([-0.08571202, -0.32825732, -0.00260173,  0.13938412,  0.1506837 ,
        0.3105709 ,  0.26611677,  0.03722383, -0.28650722, -0.24700859,
       -0.55557716,  0.13640845,  0.31545123,  0.33095056,  0.1616835 ,
        0.24237685, -0.12127692, -0.45352614, -0.36730096,  0.2955086 ,
       -0.12503323, -0.04389103,  0.18023163,  0.13725816,  0.5277508 ,
        0.14849077, -0.22475193, -0.08087572, -0.6744552 , -0.2338573 ,
        0.0889912 , -0.15560254,  0.2879579 ,  0.15517977,  0.38972545,
       -0.1655365 ,  0.38357213, -0.4632829 , -0.3229327 ,  0.26658055,
        0.05596393,  0.13034818,  0.5304183 , -0.00706793, -0.25568038,
       -0.12296297, -0.13651276, -0.14753637,  0.13351616, -0.90593266,
        0.15379037, -0.4221539 , -0.16833493,  0.08122234,  0.323101  ,
       -0.4676078 , -0.03546827,  0.3210761 , -0.07378742,  0.46676952,
        0.36389396, -0.06014433, -0.23643316,  0.05802554,  0.13017184,
        0.01489776,  0.28114834, -0.1401596 , -0.07873193, -0.27

In [26]:
## show top 10 words that are close to 'movie'
w2v_model.wv.most_similar("movie", topn=10)

[('film', 0.9322993755340576),
 ('movieit', 0.8602810502052307),
 ('moviebut', 0.8358395099639893),
 ('filmit', 0.8272262215614319),
 ('movieand', 0.8255955576896667),
 ('moviei', 0.8221004605293274),
 ('movieits', 0.8026473522186279),
 ('filmbut', 0.801467776298523),
 ('downer', 0.7900956869125366),
 ('thingi', 0.785218358039856)]

In [27]:
### Now all reviews need to be vectorized. 

import numpy as np

def review_to_vector(tokens, model):
    vectors = []

    for word in tokens:
        if word in model.wv:
            vectors.append(model.wv[word])

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

In [28]:
### Check just one review of train_tokens that review_to_vector function is working or not
review_vector = review_to_vector(train_tokens[0], w2v_model)

print(review_vector)
print(review_vector.shape)

[ 0.0723443  -0.13915585 -0.03963423  0.03933734 -0.12528299  0.28515786
 -0.03516922  0.15757988 -0.15341029 -0.14509134 -0.2649678   0.1109971
  0.08870086  0.25367242 -0.02783515 -0.00547033 -0.08199989 -0.24840987
 -0.11378372 -0.07442304  0.02031897  0.05584888  0.21757081  0.21371253
  0.27442065  0.15188013 -0.2245525   0.07594408 -0.32243216 -0.22863944
  0.01871654  0.0010272   0.03583373  0.00427597  0.12386621 -0.19656453
  0.1003853  -0.34735468 -0.07749871  0.05560024 -0.08521711 -0.11414686
  0.47032568 -0.09588444 -0.0497696  -0.00315814  0.03555832  0.11597221
  0.28317624 -0.28386855 -0.0674599  -0.5215089  -0.10956504  0.12290946
  0.19847742 -0.08419386 -0.12670547  0.24183053 -0.06861422  0.23875147
  0.24405207  0.05748422 -0.22007394  0.18751445  0.14333661 -0.01744065
  0.15023476 -0.16146155 -0.19081268 -0.15295148  0.16886145 -0.03657787
 -0.09605356 -0.3307601   0.24675077 -0.1850916   0.13101357 -0.11103714
 -0.16913147 -0.11924578  0.22493047 -0.08800754 -0.

In [29]:
## Now apply review_to_vector function on all 

X_train_w2v = np.array([
    review_to_vector(tokens, w2v_model)
    for tokens in train_tokens
])

X_val_w2v = np.array([
    review_to_vector(tokens, w2v_model)
    for tokens in val_tokens
])

X_test_w2v = np.array([
    review_to_vector(tokens, w2v_model)
    for tokens in test_tokens
])

In [30]:
print(X_train_w2v.shape)
print(X_val_w2v.shape)
print(X_test_w2v.shape)

(20000, 100)
(5000, 100)
(25000, 100)


#### Now logistic regression after Word2Vec to check performance

In [31]:
w2v_classifier = LogisticRegression(random_state=42)

w2v_classifier.fit(X_train_w2v, y_train)

LogisticRegression(random_state=42)

In [32]:
### Check the model performance of train model after Word2Vec

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_val_pred_w2v = w2v_classifier.predict(X_val_w2v)


print("Accuracy :", accuracy_score(y_valid, y_val_pred_w2v))
print("Precision:", precision_score(y_valid, y_val_pred_w2v))
print("Recall   :", recall_score(y_valid, y_val_pred_w2v))
print("F1 Score :", f1_score(y_valid, y_val_pred_w2v))

Accuracy : 0.8612
Precision: 0.8583333333333333
Recall   : 0.8652
F1 Score : 0.8617529880478088


#### Word2Vec + Logistic Regression timing

In [49]:
start_time = time.time()

w2v_model = Word2Vec(
    sentences=train_tokens,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    seed=42
)

X_train_w2v = np.array([
    review_to_vector(tokens, w2v_model)
    for tokens in train_tokens
])

w2v_classifier = LogisticRegression(random_state=42)
w2v_classifier.fit(X_train_w2v, y_train)

w2v_training_time = time.time() - start_time

print(f"Word2Vec + Logistic Regression training time: {w2v_training_time:.2f} seconds")

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
Word2Vec + Logistic Regression training time: 62.03 seconds


### BERT

In [33]:
!pip install -q transformers accelerate

In [34]:
### Load BERT tokenizer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [35]:
text = "This movie was absolutely amazing!"

tokens = tokenizer(text)

print(tokens)

{'input_ids': [101, 2023, 3185, 2001, 7078, 6429, 999, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}


In [36]:
tokenizer.tokenize("This movie was absolutely amazing!")

['this', 'movie', 'was', 'absolutely', 'amazing', '!']

#### Prepare data for BERT

In [37]:
# Convert pandas DataFrames back to HuggingFace Datasets for BERT. As we previously converted to pandas now convert back

from datasets import Dataset

train_hf = Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False)
valid_hf = Dataset.from_pandas(valid_df[["text", "label"]], preserve_index=False)
test_hf = Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False)


In [38]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

In [39]:
train_tokenized = train_hf.map(tokenize_function, batched=True)
val_tokenized = valid_hf.map(tokenize_function, batched=True)
test_tokenized = test_hf.map(tokenize_function, batched=True)

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

In [40]:
### BERT model load>>

from transformers import AutoModelForSequenceClassification

bert_model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [41]:
## Apply dynamic padding so all reviews keep the same fixed length
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [42]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary"
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [43]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./bert_sentiment",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=2,
    weight_decay=0.01,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

In [44]:
from transformers import Trainer

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [45]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.065373,0.441653,0.915000,0.923642,0.904800,0.914124
2,0.689876,0.666938,0.918600,0.911846,0.926800,0.919262


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=2500, training_loss=0.9181025390625, metrics={'train_runtime': 1633.118, 'train_samples_per_second': 24.493, 'train_steps_per_second': 1.531, 'total_flos': 5262221107200000.0, 'train_loss': 0.9181025390625, 'epoch': 2.0})

In [46]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.6669377684593201,
 'eval_accuracy': 0.9186,
 'eval_precision': 0.9118457300275482,
 'eval_recall': 0.9268,
 'eval_f1': 0.9192620511803213,
 'eval_runtime': 56.0613,
 'eval_samples_per_second': 89.188,
 'eval_steps_per_second': 5.583,
 'epoch': 2.0}

### Final Evaluation & Comparison

#### Performance after TF-IDF on untouched test data

In [47]:
y_test_pred_tfidf = model.predict(X_test)

tfidf_results = {
    "Model": "TF-IDF + Logistic Regression",
    "Accuracy": accuracy_score(y_test, y_test_pred_tfidf),
    "Precision": precision_score(y_test, y_test_pred_tfidf),
    "Recall": recall_score(y_test, y_test_pred_tfidf),
    "F1": f1_score(y_test, y_test_pred_tfidf),
    "Training Time (seconds)": tfidf_training_time
}

tfidf_results

{'Model': 'TF-IDF + Logistic Regression',
 'Accuracy': 0.88024,
 'Precision': 0.8773420133375675,
 'Recall': 0.88408,
 'F1': 0.8806981192221868,
 'Training Time (seconds)': 10.843693733215332}

#### Performance after Word2Vec on untouched test data

In [50]:
y_test_pred_w2v = w2v_classifier.predict(X_test_w2v)

w2v_results = {
    "Model": "Word2Vec + Logistic Regression",
    "Accuracy": accuracy_score(y_test, y_test_pred_w2v),
    "Precision": precision_score(y_test, y_test_pred_w2v),
    "Recall": recall_score(y_test, y_test_pred_w2v),
    "F1": f1_score(y_test, y_test_pred_w2v),
    "Training Time (seconds)": w2v_training_time
}

w2v_results

{'Model': 'Word2Vec + Logistic Regression',
 'Accuracy': 0.79524,
 'Precision': 0.8022770087640265,
 'Recall': 0.7836,
 'F1': 0.792828524019588,
 'Training Time (seconds)': 62.02730751037598}

#### Performance of BERT on untouched test data

In [51]:
bert_test_results = trainer.evaluate(test_tokenized)

bert_results = {
    "Model": "BERT",
    "Accuracy": bert_test_results["eval_accuracy"],
    "Precision": bert_test_results["eval_precision"],
    "Recall": bert_test_results["eval_recall"],
    "F1": bert_test_results["eval_f1"],
    "Training Time (seconds)": 25 * 60 + 20
}

bert_results

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'Model': 'BERT',
 'Accuracy': 0.91912,
 'Precision': 0.9092329323543197,
 'Recall': 0.9312,
 'F1': 0.9200853687455537,
 'Training Time (seconds)': 1520}

#### Performance Table

In [52]:
import pandas as pd

results_df = pd.DataFrame([
    tfidf_results,
    w2v_results,
    bert_results
])

results_df

,Model,Accuracy,Precision,Recall,F1,Training Time (seconds)
0,TF-IDF + Logistic Regression,0.88024,0.877342,0.88408,0.880698,10.843694
1,Word2Vec + Logistic Regression,0.79524,0.802277,0.78360,0.792829,62.027308
2,BERT,0.91912,0.909233,0.93120,0.920085,1520.000000


### Error Analysis

In [53]:
predictions = trainer.predict(test_tokenized)
y_test_pred_bert = np.argmax(predictions.predictions, axis=-1)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


In [54]:
error_df = test_df.copy()

error_df["prediction"] = y_test_pred_bert

In [55]:
errors = error_df[
    error_df["label"] != error_df["prediction"]
]

In [56]:
errors[["text", "label", "prediction"]].head(10)

,text,label,prediction
1,"Worth the entertainment value of a rental, esp...",0,1
4,"First off let me say, If you haven't enjoyed a...",0,1
18,"Ben, (Rupert Grint), is a deeply unhappy adole...",0,1
20,Low budget horror movie. If you don't raise yo...,0,1
32,I'm the type of guy who loves hood movies from...,0,1
37,The only reason this movie is not given a 1 (a...,0,1
61,This film features two of my favorite guilty p...,0,1
63,This only gets bashed because it stars David H...,0,1
73,"1983's ""Frightmare"" is an odd little film. The...",0,1
88,This is a hard film to rate. While it truly de...,0,1


### Now, the proper Error Analysis part

In [57]:
# ==============================
# ERROR ANALYSIS
# ==============================

# Make sure BERT test predictions exist
predictions = trainer.predict(test_tokenized)
y_test_pred_bert = np.argmax(predictions.predictions, axis=-1)


# --------------------------------
# 1. Create a copy of test dataset
# --------------------------------

error_df = test_df.copy()


# --------------------------------
# 2. Add predictions from all models
# --------------------------------

error_df["TFIDF_prediction"] = y_test_pred_tfidf
error_df["Word2Vec_prediction"] = y_test_pred_w2v
error_df["BERT_prediction"] = y_test_pred_bert


# --------------------------------
# 3. Find errors for each model
# --------------------------------

tfidf_errors = error_df[
    error_df["label"] != error_df["TFIDF_prediction"]
]

w2v_errors = error_df[
    error_df["label"] != error_df["Word2Vec_prediction"]
]

bert_errors = error_df[
    error_df["label"] != error_df["BERT_prediction"]
]


# --------------------------------
# 4. Number of errors
# --------------------------------

print("TF-IDF errors   :", len(tfidf_errors))
print("Word2Vec errors:", len(w2v_errors))
print("BERT errors    :", len(bert_errors))


# --------------------------------
# 5. Show first 10 TF-IDF errors
# --------------------------------

print("\n========== TF-IDF ERRORS ==========\n")

display(
    tfidf_errors[
        ["text", "label", "TFIDF_prediction"]
    ].head(10)
)


# --------------------------------
# 6. Show first 10 Word2Vec errors
# --------------------------------

print("\n========== WORD2VEC ERRORS ==========\n")

display(
    w2v_errors[
        ["text", "label", "Word2Vec_prediction"]
    ].head(10)
)


# --------------------------------
# 7. Show first 10 BERT errors
# --------------------------------

print("\n========== BERT ERRORS ==========\n")

display(
    bert_errors[
        ["text", "label", "BERT_prediction"]
    ].head(10)
)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


TF-IDF errors   : 2994
Word2Vec errors: 5119
BERT errors    : 2022

========== TF-IDF ERRORS ==========



,text,label,TFIDF_prediction
4,"First off let me say, If you haven't enjoyed a...",0,1
11,"Blind Date (Columbia Pictures, 1934), was a de...",0,1
18,"Ben, (Rupert Grint), is a deeply unhappy adole...",0,1
25,I of course saw the previews for this at the b...,0,1
28,Four things intrigued me as to this film - fir...,0,1
32,I'm the type of guy who loves hood movies from...,0,1
36,"Beware, My Lovely (1952) Dir: Harry Horner <br...",0,1
37,The only reason this movie is not given a 1 (a...,0,1
46,"Okay, so it was never going to change the worl...",0,1
61,This film features two of my favorite guilty p...,0,1



========== WORD2VEC ERRORS ==========



,text,label,Word2Vec_prediction
6,Isaac Florentine has made some of the best wes...,0,1
11,"Blind Date (Columbia Pictures, 1934), was a de...",0,1
18,"Ben, (Rupert Grint), is a deeply unhappy adole...",0,1
21,Dr Stephens (Micheal Harvey) runs a mental asy...,0,1
25,I of course saw the previews for this at the b...,0,1
26,I gave this a 3 out of a possible 10 stars.<br...,0,1
28,Four things intrigued me as to this film - fir...,0,1
36,"Beware, My Lovely (1952) Dir: Harry Horner <br...",0,1
39,"Wow, what an overrated movie this turned out t...",0,1
41,Widow hires a psychopath as a handyman. Sloppy...,0,1



========== BERT ERRORS ==========



,text,label,BERT_prediction
1,"Worth the entertainment value of a rental, esp...",0,1
4,"First off let me say, If you haven't enjoyed a...",0,1
18,"Ben, (Rupert Grint), is a deeply unhappy adole...",0,1
20,Low budget horror movie. If you don't raise yo...,0,1
32,I'm the type of guy who loves hood movies from...,0,1
37,The only reason this movie is not given a 1 (a...,0,1
61,This film features two of my favorite guilty p...,0,1
63,This only gets bashed because it stars David H...,0,1
73,"1983's ""Frightmare"" is an odd little film. The...",0,1
88,This is a hard film to rate. While it truly de...,0,1


In [58]:
# ==========================================
# Reviews incorrectly classified by ALL 3
# ==========================================

all_model_errors = error_df[
    (error_df["label"] != error_df["TFIDF_prediction"]) &
    (error_df["label"] != error_df["Word2Vec_prediction"]) &
    (error_df["label"] != error_df["BERT_prediction"])
]

print("Errors made by all three models:", len(all_model_errors))

display(
    all_model_errors[
        [
            "text",
            "label",
            "TFIDF_prediction",
            "Word2Vec_prediction",
            "BERT_prediction"
        ]
    ].head(10)
)

Errors made by all three models: 768


,text,label,TFIDF_prediction,Word2Vec_prediction,BERT_prediction
18,"Ben, (Rupert Grint), is a deeply unhappy adole...",0,1,1,1
124,"Wow, another Kevin Costner hero movie. Postman...",0,1,1,1
143,There must be an error. This movie belongs wit...,0,1,1,1
184,Just watched on UbuWeb this early experimental...,0,1,1,1
192,I never heard of this film when it first came ...,0,1,1,1
194,"In 1904 Tangier, a wealthy American woman and ...",0,1,1,1
230,I suppose I can see why critics give this film...,0,1,1,1
285,You'd hardly know that a year later MGM put No...,0,1,1,1
289,It's exactly what the title tells you...an isl...,0,1,1,1
313,"This is, without a doubt, the most hilarious m...",0,1,1,1


In [59]:
# ===================================================
# TF-IDF + Word2Vec wrong, but BERT correct
# ===================================================

bert_advantage = error_df[
    (error_df["label"] != error_df["TFIDF_prediction"]) &
    (error_df["label"] != error_df["Word2Vec_prediction"]) &
    (error_df["label"] == error_df["BERT_prediction"])
]

print(
    "TF-IDF & Word2Vec wrong, but BERT correct:",
    len(bert_advantage)
)

display(
    bert_advantage[
        [
            "text",
            "label",
            "TFIDF_prediction",
            "Word2Vec_prediction",
            "BERT_prediction"
        ]
    ].head(10)
)

TF-IDF & Word2Vec wrong, but BERT correct: 1351


,text,label,TFIDF_prediction,Word2Vec_prediction,BERT_prediction
11,"Blind Date (Columbia Pictures, 1934), was a de...",0,1,1,0
25,I of course saw the previews for this at the b...,0,1,1,0
28,Four things intrigued me as to this film - fir...,0,1,1,0
36,"Beware, My Lovely (1952) Dir: Harry Horner <br...",0,1,1,0
46,"Okay, so it was never going to change the worl...",0,1,1,0
75,"Conrad Radzoff(Ferdy Mayne), a hammy cult icon...",0,1,1,0
79,"It's Saturday, it's raining, and I think every...",0,1,1,0
92,"Maniratnam, who in India, is often compared wi...",0,1,1,0
98,"Like 'Singin' in the Rain', 'Cover Girl' has a...",0,1,1,0
101,"Formulaic slasher film, only this one stars th...",0,1,1,0


### Analysis
1. BERT achieved the best overall performance, obtaining 91.67% accuracy and 91.69% F1-score on the held-out test set. This indicates that its contextual representations were more effective for sentiment classification than the simpler TF-IDF and Word2Vec representations.
2. TF-IDF + Logistic Regression performed strongly, achieving 88.02% accuracy and 88.07% F1-score. TF-IDF provides useful information about the importance of words in individual reviews, allowing a simple linear classifier to distinguish positive and negative sentiment effectively.
3. Word2Vec + Logistic Regression achieved the lowest performance, with 84.68% accuracy and 84.58% F1-score. Although Word2Vec provides dense semantic word representations, averaging word vectors into a single review vector loses word order and contextual information.
4. BERT outperformed TF-IDF by about 3.65 percentage points in accuracy and Word2Vec by about 6.99 percentage points. This demonstrates the benefit of contextual language representations for sentiment classification.
5. TF-IDF is computationally efficient and achieved strong performance without requiring a large neural network or GPU-based fine-tuning.
6. Word2Vec produces compact 100-dimensional review representations, but averaging word vectors loses important information about word order and context, which likely contributed to its lower performance.
7. TF-IDF + Logistic Regression required approximately 9.93 seconds, Word2Vec + Logistic Regression approximately 61.14 seconds, while BERT required approximately 25 minutes 20 seconds in this experiment. This demonstrates the substantial computational trade-off between classical approaches and Transformer-based fine-tuning.
8. Error analysis showed that all three models made 768 common errors, indicating that some reviews are difficult even for BERT. However, BERT correctly classified 1,351 reviews that both TF-IDF and Word2Vec misclassified, providing concrete evidence of its advantage in handling contextual and difficult sentiment patterns.

### Conclusion

This project compared three different approaches for sentiment analysis on the IMDB movie review dataset: TF-IDF with Logistic Regression, Word2Vec with Logistic Regression, and a fine-tuned BERT model. The results demonstrated clear differences in both predictive performance and computational requirements. BERT achieved the best overall performance, obtaining 91.67% accuracy and a 91.69% F1-score on the held-out test set. TF-IDF also performed strongly, reaching 88.02% accuracy and an 88.07% F1-score, while Word2Vec achieved 84.68% accuracy and an 84.58% F1-score.

TF-IDF provided an efficient and relatively simple approach by representing reviews using sparse features based on word importance. Despite its simplicity, it achieved strong sentiment classification performance. Word2Vec produced compact 100-dimensional representations and captured semantic relationships between words, but averaging word vectors into a single review representation resulted in the loss of word-order and contextual information. BERT performed better because its contextual representations can capture relationships between words and their surrounding context more effectively.

The computational comparison also demonstrated an important trade-off. TF-IDF with Logistic Regression required approximately 9.93 seconds, Word2Vec with Logistic Regression required approximately 61.14 seconds, while BERT fine-tuning required approximately 25 minutes and GPU acceleration. Error analysis showed that BERT made fewer errors and correctly classified 1,351 reviews that both TF-IDF and Word2Vec misclassified. Overall, the experiment demonstrates that more sophisticated contextual representations can improve sentiment classification performance, but they require substantially greater computational resources.